# SEA robustness assessment (SM-3)

This notebook builds the SEA of the hydrometallurgical Li-ion battery recycling case study with the **same functions and the same input data** as `main_sea.ipynb`, and organises the analysis in three sections:

1. **Robustness to the selection of the original process $p_0$** (SM-3, Figure S2): the SEA stage construction is repeated for two different choices of original process, and the resulting entropy evolutions are compared.
2. **Robustness to the selection of the target elements** (SM-3, Figure S3): the SEA is recomputed for different lists of target elements (diluted in a *Rest* fraction).
3. **Product contributions of the first and last stage** (for the original process $p_0$ of option 1, `X = 0`): all natural environment output flows are grouped under an artificial flow named `elementary_flows`, and the cut-off waste input LiBs (`f_0-1_eol-libs`) is kept explicit.

# Initialisation

## 📖 required libraries

In [ ]:
# Python librairies
import numpy as np
import pandas as pd

from collections import defaultdict, deque

# For plots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D


# For chemical element recognition, we can use the mendeleev library to get a list of valid chemical symbols.
# !pip install mendeleev # Uncomment this line to install mendeleev if it's not already installed
from mendeleev import element as mendeleev_element # Importing the element function from mendeleev

## 🚴 your local paths

In [ ]:
# input path
file_path = "" #/old
excel_file_path = f"{file_path}sea.xlsx"
print(excel_file_path)

## information element formalism: the chemical elements

In [ ]:
# Function 'is_chemical_element' to check if a name is a chemical element for Statistical Entropy.
# Modified to also recognize "Rest" as a valid component.
# 1. Is used in the mass balance checking function for the process simulated input data.
# 
try:
    def is_chemical_element(name):
        # Special case: "Rest" is treated like an element for entropy calculations
        if name == "Rest":
            return True
        
        if len(name) <= 2 and name.isalpha():
            try:
                mendeleev_element(name)
                return True
            except:
                return False
        return False
except ImportError:
    # Fallback: use common chemical symbols if mendeleev not available
    chemical_symbols = {
        'H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar',
        'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr',
        'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Xe',
        'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu',
        'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn',
        'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md', 'No', 'Lr'
    }
    
    def is_chemical_element(name):
        # Special case: "Rest" is treated like an element for entropy calculations
        if name == "Rest":
            return True
        
        return 1 <= len(name) <= 2 and name in chemical_symbols

In [ ]:
df = pd.read_excel(excel_file_path)
            
# Clean column names (remove extra spaces)
df.columns = df.columns.str.strip()

# Identify element columns (they should be the ones after "Mass Flow Rate (t/h)")
all_columns = df.columns.tolist()
mass_flow_col = 'mass-FU_kg-eq' # oldversion: "Mass Flow Rate (t/h)" # newversion: "mass-FU_kg-eq"
mass_flow_idx = all_columns.index(mass_flow_col)

# Assuming element columns are all columns after the mass flow column, we can filter them using the is_chemical_element function
element_columns = [col for col in all_columns if is_chemical_element(col)]
element_columns        

# Stage diagram construction

## functions

### get the original process(es) candidate(s)

In [ ]:
def get_original_processes(excel_file):
    """
    Return list of original processes (root nodes).
    1. Processes with "fu" in any input provider
    2. Processes with no process inputs
    """
    xl = pd.ExcelFile(excel_file)
    sheet_names = xl.sheet_names
    all_processes = set(sheet_names)
    
    original_processes = []
    
    for process in sheet_names:
        df = pd.read_excel(excel_file, sheet_name=process)
        df.columns = df.columns.str.strip()
        
        has_fu = False
        has_process_input = False
        
        for _, row in df.iterrows():
            direction = str(row.get("direction", "")).strip().lower()
            provider = str(row.get("provider/consumer", "")).strip()
            
            if direction == "input":
                if "fu" in provider.lower(): # if the system modelled aims to recycle a certain amount of flow, the first stage should be the one that import the waste flow to be treated
                    has_fu = True
                if provider in all_processes:
                    has_process_input = True # if the process has at least one process input, it is not an original process, except if it also has a "fu" input
        
        if has_fu or not has_process_input:
            original_processes.append(process)
    
    return original_processes

### processes ranking

In [ ]:
def assign_process_ranks_all_options(excel_file, original_processes_list):
    """
    Assign ranks starting from EACH original process separately.
    Includes back-propagation: unranked providers get rank = min(consumer_rank) - 1.
    Returns all ranking options.
    """
    xl = pd.ExcelFile(excel_file)
    sheet_names = xl.sheet_names
    all_processes = set(sheet_names)

    # Build adjacency and store flow details (shared across all original processes)
    process_consumers = defaultdict(list)
    process_providers = defaultdict(list)
    flow_details = defaultdict(list)
    
    for sheet_name in sheet_names:
        try:
            df = pd.read_excel(excel_file, sheet_name=sheet_name)
            df.columns = df.columns.str.strip()
            
            for _, row in df.iterrows():
                flow_name = row.get("flow_name")
                flow_type = row.get("flow_type")
                consumer = row.get("provider/consumer")
                direction = row.get("direction")
                
                if pd.notna(flow_name) and pd.notna(consumer):
                    if direction == "output" and consumer in all_processes:
                        process_consumers[sheet_name].append(consumer)
                        process_providers[consumer].append(sheet_name)
                        flow_details[(sheet_name, consumer)].append({
                            "flow_name": flow_name,
                            "flow_type": flow_type,
                            "consumer": consumer
                        })
        except Exception as e:
            print(f"Error processing sheet {sheet_name}: {e}")

    # Store results for EACH original process separately
    all_rankings = {}
    
    for orig in original_processes_list:
        process_ranks = defaultdict(set)
        recycling_flows_set = set()

        # Forward BFS from original process
        queue = deque([(orig, 1)])
        
        while queue:
            current, rank = queue.popleft()

            if rank not in process_ranks[current]:
                process_ranks[current].add(rank)

            for consumer in process_consumers[current]:
                new_rank = rank + 1
                max_consumer_rank = max(process_ranks[consumer]) if process_ranks[consumer] else 0

                if max_consumer_rank and max_consumer_rank < rank:
                    for flow in flow_details.get((current, consumer), []):
                        recycling_flows_set.add((
                            current,
                            consumer,
                            flow["flow_name"],
                            flow["flow_type"]
                        ))
                    continue

                if new_rank not in process_ranks[consumer]:
                    queue.append((consumer, new_rank))

        # =====================================================================
        # NEW: Back-propagate ranks to unranked providers
        # =====================================================================
        changed = True
        while changed:
            changed = False
            for process in list(all_processes):
                if process not in process_ranks or not process_ranks[process]:
                    # This process has no rank yet
                    # Check its consumers - if any are ranked, rank this process as min(consumer_rank) - 1
                    if process in process_consumers:
                        consumer_ranks = []
                        for consumer in process_consumers[process]:
                            if consumer in process_ranks and process_ranks[consumer]:
                                consumer_ranks.append(min(process_ranks[consumer]))
                        
                        if consumer_ranks:
                            provider_rank = min(consumer_ranks) - 1
                            if provider_rank < 1:
                                provider_rank = 1  # All ranks start at 1 minimum
                            
                            if provider_rank not in process_ranks[process]:
                                process_ranks[process].add(provider_rank)
                                changed = True
            
            # Also back-propagate from newly ranked processes
            for process in list(process_ranks.keys()):
                if process_ranks[process]:
                    min_rank = min(process_ranks[process])
                    # Check providers of this process
                    for provider in process_providers.get(process, []):
                        if provider not in process_ranks or not process_ranks[provider]:
                            provider_rank = min_rank - 1
                            if provider_rank < 1:
                                provider_rank = 1
                            if provider_rank not in process_ranks[provider]:
                                process_ranks[provider].add(provider_rank)
                                changed = True
        
        # =====================================================================
        # END NEW SECTION
        # =====================================================================

        # Backfill ranks for original process
        if process_consumers[orig]:
            max_rank = max(
                (max(process_ranks[c]) for c in process_consumers[orig] if process_ranks[c]),
                default=1
            )
            if max_rank > 1:
                process_ranks[orig].update(range(1, max_rank))
        else:
            if not process_ranks[orig]:
                process_ranks[orig].add(1)

        max_rank_per_process = {p: max(ranks) for p, ranks in process_ranks.items()}
        
        recycling_flows = [
            {
                "flow_name": fn,
                "flow_type": ft,
                "producer_process": fp,
                "producer_rank": max_rank_per_process.get(fp),
                "consumer_process": tp,
                "consumer_rank": max_rank_per_process.get(tp)
            }
            for fp, tp, fn, ft in recycling_flows_set
        ]

        all_rankings[orig] = {
            "process_ranks": dict(process_ranks),
            "max_rank_per_process": max_rank_per_process,
            "recycling_flows": recycling_flows
        }
    
    return all_rankings, dict(process_consumers), dict(process_providers)


def print_all_ranking_options(all_rankings):
    """
    Print all ranking options and save each as CSV.
    Returns a list of ranking dataframes for easy access by index.
    """
    print("\n" + "="*70)
    print("ALL RANKING OPTIONS (by original process)")
    print("="*70)
    
    ranking_dfs = []  # Store dataframes for access by index
    
    for i, (orig, data) in enumerate(all_rankings.items(), 1):
        print(f"\n--- Option {i}: Starting from '{orig}' ---")
        print(f"    Number of processes ranked: {len(data['max_rank_per_process'])}")
        print(f"    Max stage: {max(data['max_rank_per_process'].values()) if data['max_rank_per_process'] else 'N/A'}")
        print(f"    Recycling flows detected: {len(data['recycling_flows'])}")
        
        # Create dataframe for this ranking
        ranking_df = pd.DataFrame.from_dict(
            data['max_rank_per_process'], 
            orient='index', 
            columns=['max_rank']
        )
        ranking_df['all_ranks'] = ranking_df.index.map(
            lambda p: sorted(data['process_ranks'][p])
        )
        ranking_df.index.name = 'process'
        ranking_df = ranking_df.reset_index()
        ranking_df['option'] = i
        ranking_df['original_process'] = orig
        
        # Save as CSV
        csv_filename = f"{file_path}ranking_option_{i}_{orig}.csv"
        ranking_df.to_csv(csv_filename, index=False)
        print(f"    → Saved: {csv_filename}")
        
        # # Show the ranking
        # for _, row in ranking_df.iterrows():
        #     print(f"      {row['process']}: ranks {row['all_ranks']} → max={row['max_rank']}")
        
        # if data['recycling_flows']:
        #     print(f"    Recycling flows:")
        #     recycling_df = pd.DataFrame(data['recycling_flows'])
        #     recycling_csv = f"{file_path}recycling_option_{i}_{orig}.csv"
        #     recycling_df.to_csv(recycling_csv, index=False)
        #     print(f"    → Saved recycling: {recycling_csv}")
        #     for rf in data['recycling_flows']:
        #         print(f"      {rf['producer_process']}(r{rf['producer_rank']}) → {rf['consumer_process']}(r{rf['consumer_rank']}) : {rf['flow_name']}")
        
        ranking_dfs.append({
            'option_num': i,
            'original_process': orig,
            'ranking_df': ranking_df,
            'max_rank_per_process': data['max_rank_per_process'],
            'process_ranks': data['process_ranks'],
            'recycling_flows': data['recycling_flows']
        })
    
    return all_rankings, ranking_dfs

def select_ranking_option(all_rankings):
    """
    Let user choose which ranking option to use, or auto-select if only one.
    """
    originals = list(all_rankings.keys())
    
    if len(originals) == 1:
        chosen = originals[0]
        print(f"\n✓ Only one original process found. Auto-selecting: '{chosen}'")
    else:
        print(f"\n{'='*50}")
        print("SELECT RANKING OPTION")
        print(f"{'='*50}")
        for i, orig in enumerate(originals, 1):
            print(f"  {i}. Start from '{orig}' ({len(all_rankings[orig]['max_rank_per_process'])} processes)")
        
        while True:
            try:
                choice = int(input(f"\nEnter number (1-{len(originals)}): "))
                if 1 <= choice <= len(originals):
                    chosen = originals[choice - 1]
                    break
                else:
                    print(f"Please enter a number between 1 and {len(originals)}")
            except ValueError:
                print("Please enter a valid number")
    
    print(f"\n→ Selected: '{chosen}'")
    return chosen, all_rankings[chosen]

### flows distinctions

In [ ]:

def get_internal_recycling_flows(
    excel_file,
    process_consumers,
    max_rank_per_process
):
    """
    Detect internal recycling flows.

    Recycling condition:
    producer max rank >= consumer max rank
    """

    xl = pd.ExcelFile(excel_file)
    sheet_names = xl.sheet_names
    all_processes = set(sheet_names)

    recycling_flows = []

    for producer_process in sheet_names:

        df = pd.read_excel(excel_file, sheet_name=producer_process)
        df.columns = df.columns.str.strip()

        for _, row in df.iterrows():

            direction = row.get("direction")
            consumer_process = row.get("provider/consumer")

            if (
                direction == "output"
                and pd.notna(consumer_process)
                and consumer_process in all_processes
            ):

                producer_rank = max_rank_per_process.get(producer_process)
                consumer_rank = max_rank_per_process.get(consumer_process)

                # Recycling criterion
                if producer_rank is not None and consumer_rank is not None and producer_rank >= consumer_rank:

                    recycling_flows.append({
                        "flow_name": row.get("flow_name"),
                        "flow_type": row.get("flow_type"),

                        "producer_process": producer_process,
                        "producer_rank": producer_rank,

                        "consumer_process": consumer_process,
                        "consumer_rank": consumer_rank
                    })

    return pd.DataFrame(recycling_flows)


# =============================================================================
# 5. INPUT FLOWS FROM ENVIRONMENT
# =============================================================================

def get_input_environment_flows(
    excel_file,
    max_rank_per_process
):
    """
    Input flows coming from environment, background database (as for LCA), or stocks.
    """

    xl = pd.ExcelFile(excel_file)

    env_inputs = []

    for process in xl.sheet_names:

        df = pd.read_excel(excel_file, sheet_name=process)
        df.columns = df.columns.str.strip()

        for _, row in df.iterrows():

            direction = str(row.get("direction")).strip().lower()
            provider = str(row.get("provider/consumer")).lower()

            if (
                direction == "input"
                and (
                    "environment" in provider
                    or "background" in provider
                    or "stock" in provider
                )
            ):

                env_inputs.append({

                    "flow_name": row.get("flow_name"),
                    "flow_type": row.get("flow_type"),

                    "consumer_process": process,
                    "consumer_rank": max_rank_per_process.get(process),

                    "provider": row.get("provider/consumer")
                })

    return pd.DataFrame(env_inputs)


# =============================================================================
# 6. OUTPUT FLOWS TO ENVIRONMENT
# =============================================================================

def get_output_environment_flows(
    excel_file,
    max_rank_per_process
):
    """
    Output flows going to environment/background.
    """

    xl = pd.ExcelFile(excel_file)

    env_outputs = []

    for process in xl.sheet_names:

        df = pd.read_excel(excel_file, sheet_name=process)
        df.columns = df.columns.str.strip()

        for _, row in df.iterrows():

            direction = str(row.get("direction")).strip().lower()
            consumer = str(row.get("provider/consumer")).lower()

            if (
                direction == "output"
                and (
                    "environment" in consumer
                    or "background" in consumer
                    or "stock" in consumer
                )
            ):

                env_outputs.append({

                    "flow_name": row.get("flow_name"),
                    "flow_type": row.get("flow_type"),

                    "producer_process": process,
                    "producer_rank": max_rank_per_process.get(process),

                    "consumer": row.get("provider/consumer")
                })

    return pd.DataFrame(env_outputs)


# =============================================================================
# 7. INTERMEDIATE FOREGROUND PRODUCT FLOWS
# =============================================================================

def get_intermediate_foreground_product_flows(
    excel_file,
    max_rank_per_process,
    recycling_flows_df
):
    """
    Intermediate foreground product flows.

    These are:
    - input flows
    - provider is another foreground process
    - NOT recycling flows
    - NOT environment, background, or from or to a stock flows
    """

    xl = pd.ExcelFile(excel_file)

    all_processes = set(xl.sheet_names)

    # Recycling flow names
    # Recycling flow names
    if recycling_flows_df.empty:
        recycling_flow_names = set()
    else:
        recycling_flow_names = set(recycling_flows_df["flow_name"])

    foreground_flows = []

    for consumer_process in xl.sheet_names:

        df = pd.read_excel(excel_file, sheet_name=consumer_process)
        df.columns = df.columns.str.strip()

        for _, row in df.iterrows():

            direction = str(row.get("direction")).strip().lower()
            provider = row.get("provider/consumer")
            flow_name = row.get("flow_name")

            if pd.isna(provider):
                continue

            provider_str = str(provider).lower()

            if (
                direction == "input"
                and provider in all_processes
                and "environment" not in provider_str
                and "stock" not in provider_str
                and "background" not in provider_str
                and flow_name not in recycling_flow_names
            ):

                foreground_flows.append({

                    "flow_name": flow_name,
                    "flow_type": row.get("flow_type"),

                    "provider_process": provider,
                    "provider_rank": max_rank_per_process.get(provider),

                    "consumer_process": consumer_process,
                    "consumer_rank": max_rank_per_process.get(consumer_process)
                })

    return pd.DataFrame(foreground_flows)

### stages creation

In [ ]:
# =============================================================================
# 1. SYSTEM STAGE DIAGRAM
# =============================================================================

def build_system_2stage_dataframes(
    excel_file,
    max_rank_per_process,
    input_env_flows_df,
    output_env_flows_df
):
    """
    Build 2-stage system diagram with element columns.
    Stage 1: ALL environment inputs to the system (from outside)
    Stage 2: ALL environment outputs from the system (to outside)
    """
    xl = pd.ExcelFile(excel_file)
    ranked_processes = set(max_rank_per_process.keys())
    
    # Load all flows once to get element data
    sheet_names = xl.sheet_names
    df_all_flows = pd.concat(
        [pd.read_excel(excel_file, sheet_name=s).assign(_sheet=s) for s in sheet_names],
        ignore_index=True
    )
    df_all_flows.columns = df_all_flows.columns.str.strip()
    
    # Auto-detect element columns
    element_cols = [col for col in df_all_flows.columns if is_chemical_element(col)]
    columns_to_include = element_cols + ['mass-FU_kg-eq']
    
    stage1_flows = []
    stage2_flows = []
    
    for process in ranked_processes:
        try:
            df = pd.read_excel(excel_file, sheet_name=process)
            df.columns = df.columns.str.strip()
            
            for _, row in df.iterrows():
                flow_name = row.get("flow_name")
                provider_consumer = row.get("provider/consumer")
                direction = str(row.get("direction", "")).strip().lower()
                
                if pd.isna(flow_name) or pd.isna(provider_consumer):
                    continue
                
                provider_consumer_str = str(provider_consumer)
                
                # Input from outside the system
                if direction == "input" and provider_consumer_str not in ranked_processes:
                    stage1_flows.append({
                        "flow_name": flow_name,
                        "flow_type": row.get("flow_type"),
                        "from_process": provider_consumer_str,
                        "to_process": process,
                        **{col: row.get(col) for col in columns_to_include if col in row.index}
                    })
                
                # Output to outside the system
                if direction == "output" and provider_consumer_str not in ranked_processes:
                    stage2_flows.append({
                        "flow_name": flow_name,
                        "flow_type": row.get("flow_type"),
                        "from_process": process,
                        "to_process": provider_consumer_str,
                        **{col: row.get(col) for col in columns_to_include if col in row.index}
                    })
        
        except Exception as e:
            print(f"Error processing sheet {process}: {e}")
    
    stage_dfs = {
        1: pd.DataFrame(stage1_flows) if stage1_flows else pd.DataFrame(),
        2: pd.DataFrame(stage2_flows) if stage2_flows else pd.DataFrame()
    }
    
    return stage_dfs

# =============================================================================
# 2. NORMAL STAGE DIAGRAM
# =============================================================================
# Determination of Stages
# "The number of stages in the system is defined as one more than the number of processes. If the system contains n_P processes, the number of stages n_S is given by:
# n_S = n_P + 1
# Each stage is indexed as j = 1, 2, ..., n_S.
# The entire system can be conceptualized as a single process that sequentially transfers inputs through these stages. Each stage is characterized by a distinct set of material flows (see Fig. 3b for reference).
# "
# 📖 Citation
# Rechberger, H., & Graedel, T. E. (2002). The contemporary European copper cycle: statistical entropy analysis. Center for Industrial Ecology, School of Forestry and Environmental Studies, Yale University. Available online 6 May 2002


def build_stage_flow_dataframes(
    excel_file,
    process_ranks,
    max_rank_per_process,
    recycling_flows_df,
    input_env_flows_df,
    output_env_flows_df,
    intermediate_foreground_flows_df
):
    """
    Build dataframes for each stage containing all flows that appear in that stage.
    
    📖 Following Rechberger & Graedel (2002) and Laner et al. (2017) for flow assignment to stages, with corrections for recycling flows.
    
    ⚠️ Stocks are not treated there since not present in the model. According to Rechberger and Graedel: "Flows out of a stock are treated as input flows into the process and flows into a stock are regarded as output flows of the process."
    """
    xl = pd.ExcelFile(excel_file)
    sheet_names = xl.sheet_names
    
    # Load all sheets once
    df_all_flows = pd.concat(
        [pd.read_excel(excel_file, sheet_name=s).assign(_sheet=s) for s in sheet_names],
        ignore_index=True
    )
    df_all_flows.columns = df_all_flows.columns.str.strip()
    
    # Auto-detect element columns
    element_cols = [col for col in df_all_flows.columns if is_chemical_element(col)]
    columns_to_include = element_cols + ["mass-FU_kg-eq"]
    
    # Max stage = maximum rank + 1
    max_rank = max(max_rank_per_process.values()) if max_rank_per_process else 0
    max_stage = max_rank + 1
    
    stage_flows = {stage: [] for stage in range(1, max_stage + 1)}
    
    def add_flow_to_stage(flow_dict, stage, flow_data=None):
        if stage not in stage_flows:
            return
        record = {
            "flow_name": flow_dict.get("flow_name"),
            "flow_type": flow_dict.get("flow_type"),
            "from_process": flow_dict.get("producer_process", flow_dict.get("from_process", flow_dict.get("provider"))),
            "to_process": flow_dict.get("consumer_process", flow_dict.get("to_process", flow_dict.get("consumer"))),
        }
        if flow_data is not None:
            for col in columns_to_include:
                record[col] = flow_data.get(col)
        else:
            flow_name = flow_dict.get("flow_name")
            fd = df_all_flows[df_all_flows["flow_name"] == flow_name]
            if not fd.empty:
                for col in columns_to_include:
                    record[col] = fd.iloc[0].get(col)
        record['stage'] = stage
        if not any(f["flow_name"] == record["flow_name"] and f.get("from_process") == record["from_process"] and f.get("to_process") == record["to_process"] for f in stage_flows[stage]):
            stage_flows[stage].append(record)
    
    # 1. Input environment flows: stages 1 to consumer_rank
    for _, row in input_env_flows_df.iterrows():
        fd = row.to_dict()
        rank = max_rank_per_process.get(fd.get('consumer_process'))
        if rank:
            flow_data = df_all_flows[df_all_flows["flow_name"] == fd.get('flow_name')]
            flow_data = flow_data.iloc[0] if not flow_data.empty else None
            for stage in range(1, rank + 1):
                add_flow_to_stage(fd, stage, flow_data)
    
    # 2. Intermediate foreground flows: provider_rank+1 to consumer_rank
    for _, row in intermediate_foreground_flows_df.iterrows():
        fd = row.to_dict()
        pr, cr = fd.get('provider_rank'), fd.get('consumer_rank')
        if pr and cr:
            flow_data = df_all_flows[df_all_flows["flow_name"] == fd.get('flow_name')]
            flow_data = flow_data.iloc[0] if not flow_data.empty else None
            for stage in range(pr + 1, cr + 1):
                add_flow_to_stage(fd, stage, flow_data)
    
    # 3. Output environment flows: stages after producer_rank to max_stage
    for _, row in output_env_flows_df.iterrows():
        fd = row.to_dict()
        pr = fd.get('producer_rank')
        if pr:
            flow_data = df_all_flows[df_all_flows["flow_name"] == fd.get('flow_name')]
            flow_data = flow_data.iloc[0] if not flow_data.empty else None
            for stage in range(pr + 1, max_stage + 1):
                add_flow_to_stage(fd, stage, flow_data)
    
    # 4. Recycling flows: producer_rank+1, consumer_rank, and outside range
    for _, row in recycling_flows_df.iterrows():
        fd = row.to_dict()
        pr, cr = fd.get('producer_rank'), fd.get('consumer_rank')
        if pr and cr:
            flow_data = df_all_flows[df_all_flows["flow_name"] == fd.get('flow_name')]
            flow_data = flow_data.iloc[0] if not flow_data.empty else None
            start_range, end_range = min(cr, pr), max(cr, pr)
            fd['recycling_direction'] = 'backward' if pr > cr else 'forward'
            for stage in range(1, max_stage + 1):
                if stage == pr + 1 or stage == cr or stage < start_range or stage > end_range:
                    add_flow_to_stage(fd, stage, flow_data)
    
    # Convert to DataFrames
    stage_dfs = {}
    col_order = ['stage', 'flow_name', 'flow_type', 'from_process', 'to_process']
    
    for stage in sorted(stage_flows.keys()):
        if stage_flows[stage]:
            df = pd.DataFrame(stage_flows[stage])
            order = col_order.copy()
            if 'recycling_direction' in df.columns:
                order.insert(3, 'recycling_direction')
            order += [c for c in columns_to_include if c in df.columns]
            df = df[[c for c in order if c in df.columns]].sort_values('flow_name').reset_index(drop=True)
            stage_dfs[stage] = df
        else:
            stage_dfs[stage] = pd.DataFrame()
    
    return stage_dfs

# Compute the SEA

Now, we have all the `stage_dfs[i]` for `i` going from `1` to the number of stages of the flowsheet


## for one stage

The `compute_stage_entropy` function updates the `stage_dfs[i]`: 3 columns added. One for the flows Shannon, Rechberger and Relative Statistical Entropy (RSE) information values.

In [ ]:
import numpy as np

def compute_stage_entropy(stage_df):
    """
    Compute Shannon, Rechberger, and Relative entropy for each flow in a stage.
    Add these values for each flow of a stage in the stage_dfs dataframe.
    """
    df = stage_df.copy()

    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    mass_idx = df.columns.get_loc("mass-FU_kg-eq")
    element_cols = [col for col in df.columns if is_chemical_element(col)]
    
    # Total mass of the stage
    M = df["mass-FU_kg-eq"].sum()

    # Number of elements considered (for relative entropy)
    n_R = len(element_cols)

    # Prepare accumulators
    shannon_vals = []
    rechberger_vals = []
    relative_vals = []

    for _, row in df.iterrows():
        m_j = row["mass-FU_kg-eq"]

        # Initialize entropies for this flow
        H_shannon = 0.0
        H_rechberger = 0.0
        H_relative = 0.0

        for col in element_cols:
            c = row[col]
            if pd.isna(c) or c <= 0:
                continue

            p = c / 100.0  # convert to fraction

            # Shannon
            H_shannon += - p * np.log2(p)

            # Rechberger
            H_rechberger += - (m_j / M) * p * np.log2(p)

            # Relative
            H_relative += - (m_j / M) * p * (np.log(p) / np.log(n_R))  # log base n_R

        shannon_vals.append(H_shannon)
        rechberger_vals.append(H_rechberger)
        relative_vals.append(H_relative)

    # Add new columns
    df["Shannon entropy per flow"] = shannon_vals
    df["Rechberger entropy per flow"] = rechberger_vals
    df["Relative entropy per flow"] = relative_vals

    return df

## for elements

In [ ]:
def get_unique_elements(stage_dfs):
    # Exclude non-element columns
    exclude_columns = ['flow_name', 'flow_type', 'from_process', 'to_process', 'mass-FU_kg-eq']
    elements = set()

    for stage, df in stage_dfs.items():
        for col in df.columns:
            if col not in exclude_columns:
                elements.add(col)

    return sorted(list(elements))
    
def group_elements_of_interest(stage_dfs, elements_of_interest):
    """
    Parameters:
    - Require the stages analysis to be done.
    - Require a list of elements of interest.
    
    Actions:
    - Modify stage_dfs to focus only on the list of elements for which we want to measure the contribution to the Statistical Entropies value.
    - A rest column replaces the sum of all the other elements present in the flowsheet.

    Return:
    - a modified restricted stage_dfs dataframe, with the 'elements_of_interest' and the rest fraction.
    """
    modified_stage_dfs = {}

    for stage, df in stage_dfs.items():
        df = df.copy()
        # Identify element columns: all columns after "mass-FU_kg-eq"
        if "mass-FU_kg-eq" not in df.columns:
            raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
        mass_idx = df.columns.get_loc("mass-FU_kg-eq")
        element_cols = [col for col in df.columns if is_chemical_element(col)]

        # Create a new column for the rest
        rest_col = "Rest"
        df[rest_col] = 100  # Initialize to 100%

        # Subtract the elements of interest from the rest
        for element in elements_of_interest:
            if element in element_cols:
                df[rest_col] -= df[element].fillna(0)

        # Create a new DataFrame with only the elements of interest and the rest
        new_element_cols = elements_of_interest + [rest_col]
        new_element_cols = [col for col in new_element_cols if col in df.columns]

        # Keep only the relevant columns
        relevant_cols = [col for col in df.columns if col not in element_cols or col in new_element_cols]
        modified_df = df[relevant_cols]

        modified_stage_dfs[stage] = modified_df

    return modified_stage_dfs

def compute_element_entropy_across_stages(stage_dfs):
    """
    Parameters:
    - Require the stages analysis to be done.

    Actions:
    - Compute the statistical entropy per element across all stages.

    Return:
    - A dataframe with all the flowsheet's elements (rows) and their corresponding SE and RSE (2 columns) for each stage.
    """
    element_entropy = {stage: {} for stage in stage_dfs}

    for stage, df in stage_dfs.items():
        # Identify element columns: all columns after "mass-FU_kg-eq"
        if "mass-FU_kg-eq" not in df.columns:
            raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
        mass_idx = df.columns.get_loc("mass-FU_kg-eq")
        element_cols = [col for col in df.columns if is_chemical_element(col)]
        M = df["mass-FU_kg-eq"].sum() # sum of all stage mass flow
        n_R = len(element_cols) # number of elements for normalisation (get the RSE)

        for col in element_cols:
            H_rechberger = 0.0
            H_relative = 0.0

            for _, row in df.iterrows():
                c = row[col]
                m_j = row["mass-FU_kg-eq"]
                if pd.isna(c) or c <= 0:
                    continue
                p = c / 100.0
                H_rechberger += -(m_j / M) * p * np.log2(p)
                H_relative += -(m_j / M) * p * (np.log(p) / np.log(n_R))

            element_entropy[stage][col] = {
                "Rechberger": H_rechberger,
                "Relative": H_relative
            }

    # Convert to DataFrames
    element_entropy_dfs = {}
    for stage, elements in element_entropy.items():
        rows = []
        for col, values in elements.items():
            row = {
                "Element": col,
                "Rechberger": values["Rechberger"],
                "Relative": values["Relative"]
            }
            rows.append(row)
        element_entropy_dfs[stage] = pd.DataFrame(rows).set_index("Element")

    return element_entropy_dfs

def get_element_contributions_per_flow(stage_df, flow_name):
    """
    Parameters:
    - Require a stage dataframe, and the name of one of its flows to analyse. The stage dataframe may be modified by the 'group_elements_of_interest' function.

    Actions:
    - Compute the SE and the RSE value (normalised contribution) attributed to each element that constitutes this flow.

    Return:
    - A pandas DataFrame with element names, Rechberger (SE) contributions, and Relative (RSE) contributions
    """
    
    df = stage_df.copy()
    
    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    
    # Get element columns
    all_cols = df.columns.tolist()
    element_cols = [col for col in all_cols if is_chemical_element(col)]
    
    if not element_cols:
        raise ValueError("No element columns found in the dataframe")
    
    # Filter for the specific flow
    flow_matches = df[df["flow_name"] == flow_name]
    if flow_matches.empty:
        raise ValueError(f"Flow '{flow_name}' not found in the dataframe")
    
    flow_data = flow_matches.iloc[0]

    # Calculate total entropy for the flow
    M = df["mass-FU_kg-eq"].sum() 
    m_j = flow_data["mass-FU_kg-eq"]
    n_R = len(element_cols)

    total_rechberger = 0.0
    total_relative = 0.0

    results = []

    for col in element_cols:
        c = flow_data[col]
        if pd.isna(c) or c <= 0:
            results.append({
                'element': col,
                'rechberger_contribution': 0,
                'relative_contribution': 0,
                'rechberger_normalized': 0,
                'relative_normalized': 0
            })
            continue
        
        p = c / 100.0  # Convert percentage to fraction
        rechberger_contribution = -(m_j / M) * p * np.log2(p) if p > 0 else 0
        relative_contribution = -(m_j / M) * p * (np.log(p) / np.log(n_R)) if p > 0 and n_R > 1 else 0

        total_rechberger += rechberger_contribution
        total_relative += relative_contribution

        results.append({
            'element': col,
            'rechberger_contribution': rechberger_contribution,
            'relative_contribution': relative_contribution,
            'rechberger_normalized': 0,  # Will update after total is known
            'relative_normalized': 0      # Will update after total is known
        })

    # Normalize contributions and update results
    for result in results:
        if total_rechberger != 0:
            result['rechberger_normalized'] = result['rechberger_contribution'] / total_rechberger
        if total_relative != 0:
            result['relative_normalized'] = result['relative_contribution'] / total_relative

    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Add flow name and mass info as attributes
    results_df.attrs['flow_name'] = flow_name
    results_df.attrs['total_mass'] = M
    results_df.attrs['flow_mass'] = m_j
    
    return results_df

def plot_element_contributions_per_flow(stage_df, flow_name):
    """
    Parameters:
    - Require a stage dataframe, and the name of one of its flows to analyse. The stage dataframe may be modified by the 'group_elements_of_interest' function.

    Actions:
    - Compute the SE and the RSE value (normalised contribution) attributed to each element that constitutes this flow.

    Return:
    - A bar plot: one bar per element and their SE and RSE value along the y-axis.
    """
    
    df = stage_df.copy()
    
    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    mass_idx = df.columns.get_loc("mass-FU_kg-eq")
    element_cols = [col for col in df.columns if is_chemical_element(col)]

    # Filter for the specific flow
    flow_data = df[df["flow_name"] == flow_name].iloc[0]

    # Calculate total entropy for the flow
    M = df["mass-FU_kg-eq"].sum() 
    m_j = flow_data["mass-FU_kg-eq"]
    n_R = len(element_cols)

    total_rechberger = 0.0
    total_relative = 0.0

    contributions_rechberger = []
    contributions_relative = []

    for col in element_cols:
        c = flow_data[col]
        if pd.isna(c) or c <= 0:
            contributions_rechberger.append(0)
            contributions_relative.append(0)
            continue
        p = c / 100.0
        rechberger_contribution = -(m_j / M) * p * np.log2(p)
        relative_contribution = -(m_j / M) * p * (np.log(p) / np.log(n_R))

        total_rechberger += rechberger_contribution
        total_relative += relative_contribution

        contributions_rechberger.append(rechberger_contribution)
        contributions_relative.append(relative_contribution)

    # Normalize contributions
    normalized_rechberger = [contribution / total_rechberger if total_rechberger != 0 else 0 for contribution in contributions_rechberger]
    normalized_relative = [contribution / total_relative if total_relative != 0 else 0 for contribution in contributions_relative]

    # Plot contributions
    x = np.arange(len(element_cols))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 8))
    rects2 = ax.bar(x, normalized_rechberger, width, label='Rechberger')
    rects3 = ax.bar(x + width, normalized_relative, width, label='Relative')

    ax.set_ylabel('Normalized Contribution to Entropy')
    ax.set_title(f'Normalized Element Contributions to Entropy for Flow: {flow_name}')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(element_cols, rotation=90)
    ax.legend()

    fig.tight_layout()
    
    # plt.savefig(f'{file_path}element_contributions_per_flow_{flow_name.replace("/", "_").replace(",", "_").replace(" ", "_")}.svg', dpi=200, bbox_inches='tight')
   
    plt.show()  # Display the plot interactively
    # plt.close(fig)  # Close the figure after showing it
    
def plot_an_element_contributions_for_stage(stage_df, element):
    """
    Parameters:
    - Require a stage dataframe, and the knowledge of a chemical element to analyse.

    Actions:
    - Compute the SE and RSE value (normalised contribution) of the element in each flow of the stage.

    Return:
    - Plot the normalised contribution of ONE specific element to the entropy of all flows in ONE stage.
    """
    df = stage_df.copy()
    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    mass_idx = df.columns.get_loc("mass-FU_kg-eq")
    element_cols = [col for col in df.columns if is_chemical_element(col)]

    if element not in element_cols:
        print(f"Element '{element}' not found in the stage data. Available elements: {element_cols}")
        return

    M = df["mass-FU_kg-eq"].sum() # total mass of the stage
    n_R = len(element_cols)

    contributions_rechberger = []
    contributions_relative = []
    flow_names = []
    total_rechberger_entropies = []
    total_relative_entropies = []

    for _, row in df.iterrows():
        flow_name = row["flow_name"] 
        flow_names.append(flow_name)

        c = row[element]
        m_j = row["mass-FU_kg-eq"]

        total_rechberger = 0.0 
        total_relative = 0.0

        for col in element_cols:
            c_total = row[col]
            if pd.isna(c_total) or c_total <= 0:
                continue
            p_total = c_total / 100.0
            total_rechberger += -(m_j / M) * p_total * np.log2(p_total)
            total_relative += -(m_j / M) * p_total * (np.log(p_total) / np.log(n_R))

        total_rechberger_entropies.append(total_rechberger)
        total_relative_entropies.append(total_relative)

        if pd.isna(c) or c <= 0:
            contributions_rechberger.append(0)
            contributions_relative.append(0)
            continue

        p = c / 100.0
        rechberger_contribution = -(m_j / M) * p * np.log2(p)
        relative_contribution = -(m_j / M) * p * (np.log(p) / np.log(n_R))

        # Normalise contributions, i.e., divide the element's contribution by the total entropy of the flow.
        normalized_rechberger = rechberger_contribution / total_rechberger if total_rechberger != 0 else 0 # This gives the fraction (or percentage) of the total SE that is due to an element
        normalized_relative = relative_contribution / total_relative if total_relative != 0 else 0 # This gives the fraction (or percentage) of the total RSE that is due to an element

        # Stores the normalized contributions for later plotting.
        contributions_rechberger.append(normalized_rechberger)
        contributions_relative.append(normalized_relative)

    x = np.arange(len(flow_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(14, 8))
    rects2 = ax.bar(x, contributions_rechberger, width, label='Rechberger')
    rects3 = ax.bar(x + width, contributions_relative, width, label='Relative')

    ax.set_ylabel('Normalized Contribution to Entropy')
    ax.set_title(f'Normalized Contribution of {element} to Entropy for All Flows in Stage')
    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(flow_names, rotation=90)
    ax.legend()

    fig.tight_layout()
    plt.savefig(f'{file_path}element_contribution_plot_{element}.svg', dpi=200, bbox_inches='tight')
    plt.show()

def plot_elements_contributions_for_stage(stage_df, elements, min_contribution=0.01):
    """
    Parameters:
    - stage_df: DataFrame containing the stage data.
    - elements: List of chemical elements to analyze.
    - min_contribution: Minimum contribution (as a fraction) to display the percentage on the bar.

    Actions:
    - Compute the RSE value (normalized contribution) of each element in each flow of the stage.
    - Plot the normalized contribution of the specified elements to the entropy of all flows in the stage.

    Return:
    - Plot with one stacked bar per flow, showing the percentage of each element's contribution.
    """
    df = stage_df.copy()
    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    mass_idx = df.columns.get_loc("mass-FU_kg-eq")
    element_cols = [col for col in df.columns if is_chemical_element(col)]

    # Check if all elements are in the dataframe
    missing_elements = [elem for elem in elements if elem not in element_cols]
    if missing_elements:
        print(f"Elements {missing_elements} not found in the stage data. Available elements: {element_cols}")
        return

    M = df["mass-FU_kg-eq"].sum()  # total mass of the stage
    n_R = len(element_cols)

    # Initialize dictionaries to store contributions for each element
    contributions_relative = {elem: [] for elem in elements}
    flow_names = []

    for _, row in df.iterrows():
        flow_name = row["flow_name"]
        flow_names.append(flow_name)

        m_j = row["mass-FU_kg-eq"]

        # Calculate total relative entropy for the flow
        total_relative = 0.0

        for col in element_cols:
            c_total = row[col]
            if pd.isna(c_total) or c_total <= 0:
                continue
            p_total = c_total / 100.0
            total_relative += -(m_j / M) * p_total * (np.log(p_total) / np.log(n_R))

        # Calculate contributions for each element
        for elem in elements:
            c = row[elem]
            if pd.isna(c) or c <= 0:
                contributions_relative[elem].append(0)
                continue

            p = c / 100.0
            relative_contribution = -(m_j / M) * p * (np.log(p) / np.log(n_R))

            # Normalize contributions
            normalized_relative = relative_contribution / total_relative if total_relative != 0 else 0
            contributions_relative[elem].append(normalized_relative)

    # Plotting
    x = np.arange(len(flow_names))
    width = 0.8  # Single bar per flow

    fig, ax = plt.subplots(figsize=(14, 8))

    # Initialize bottom for stacking
    bottom = np.zeros(len(flow_names))

    # Plot each element's contributions as a stacked bar
    for elem in elements:
        bars = ax.bar(x, contributions_relative[elem], width, bottom=bottom, label=elem)
        bottom += contributions_relative[elem]

        # Add percentage labels if contribution is above the threshold
        for i, contribution in enumerate(contributions_relative[elem]):
            if contribution > min_contribution:
                ax.text(x[i], bottom[i] - contribution / 2, f"{contribution:.1%}",
                        ha='center', va='center', color='black', fontsize=8)

    ax.set_ylabel('Normalized Contribution to Relative Entropy')
    ax.set_title(f'Normalized Contribution of Elements to Relative Entropy for All Flows in Stage')
    ax.set_xticks(x)
    ax.set_xticklabels(flow_names, rotation=90)
    ax.set_ylim(0, 1.1)  # Set y-axis scale from 0 to 1.1
    ax.legend()

    fig.tight_layout()
    plt.savefig(f'{file_path}element_contribution_plot_{"_".join(elements)}.svg', dpi=200, bbox_inches='tight')
    plt.show()

def plot_stacked_elements_rse_contributions_for_stage(stage_df, stage_key=None):
    """
    Parameters:
    - Require the choice of one stage to analyse, ideally a stage with a high relative statistical entropy value.

    Return:
    - Plot the stacked RSE contribution of all the flowsheet elements to the entropy of all flows in ONE stage.
    - Each element is represented by a different colour within each bar.
    """
    df = stage_df.copy()
    # Identify element columns: all columns after "mass-FU_kg-eq"
    if "mass-FU_kg-eq" not in df.columns:
        raise ValueError("Stage DataFrame must have 'mass-FU_kg-eq' column")
    mass_idx = df.columns.get_loc("mass-FU_kg-eq")
    element_cols = [col for col in df.columns if is_chemical_element(col)]
    
    M = df["mass-FU_kg-eq"].sum()
    n_R = len(element_cols)
    flow_names = df["flow_name"].tolist()

    # Create a list of colors for each element
    colors = plt.cm.tab20.colors  # type: ignore
    if len(element_cols) > len(colors):
        colors *= (len(element_cols) // len(colors) + 1)

    # Prepare data for plotting
    contributions = {element: [] for element in element_cols}
    total_rse_values = []

    for _, row in df.iterrows():
        m_j = row["mass-FU_kg-eq"]
        total_relative = 0.0

        for col in element_cols:
            c = row[col]
            if pd.isna(c) or c <= 0:
                continue
            p = c / 100.0
            total_relative += -(m_j / M) * p * (np.log(p) / np.log(n_R))

        total_rse_values.append(total_relative)

        for col in element_cols:
            c = row[col]
            if pd.isna(c) or c <= 0:
                contributions[col].append(0)
                continue
            p = c / 100.0
            relative_contribution = -(m_j / M) * p * (np.log(p) / np.log(n_R))
            contributions[col].append(relative_contribution)

    # Convert contributions to a numpy array for easier manipulation
    contributions_array = np.array([contributions[col] for col in element_cols])

    # Plot contributions as a stacked bar plot
    fig, ax = plt.subplots(figsize=(14, 14))
    x = np.arange(len(flow_names))
    ax.set_ylabel('RSE Contribution', fontsize=14)
    ax.tick_params(axis='both', which='major', labelsize=14)

    bottom = np.zeros(len(flow_names))
    for i, element in enumerate(element_cols):
        ax.bar(x, contributions_array[i], bottom=bottom, label=element, color=colors[i])
        bottom += contributions_array[i]

    ax.set_ylabel('RSE Contribution')
    # Set the title after ax is created
    if stage_key is not None:
        ax.set_title(f'Stacked RSE Contribution of All Elements for All Flows in Stage {stage_key}')
    else:
        ax.set_title('Stacked RSE Contribution of All Elements for All Flows in Stage')

    ax.set_xticks(x)
    ax.set_xticklabels(flow_names, rotation=90)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)

    fig.tight_layout()
    plt.savefig(f'{file_path}stacked_elements_rse_contributions_stage_{stage_key}.svg', dpi=200, bbox_inches='tight')
    plt.show()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable


# =============================================================================
# ELEMENT COLOR MAPPING
# =============================================================================

# Define a fixed color palette for common elements
ELEMENT_COLORS = {
    # Alkali metals
    'Li': '#ff9999',  # Light red
    'Na': '#ff6666',  # Red
    'K': '#ff3333',   # Darker red
    'Rb': '#ff0000',  # Pure red
    'Cs': '#cc0000',  # Deep red
    'Fr': '#990000',  # Dark red
    
    # Alkaline earth metals
    'Be': '#ffcc99',  # Light orange
    'Mg': '#ffb366',  # Orange
    'Ca': '#ff9933',  # Medium orange
    'Sr': '#ff8000',  # Dark orange
    'Ba': '#ff6600',  # Deep orange
    'Ra': '#cc5200',  # Brownish orange
    
    # Transition metals
    'Fe': '#99ccff',  # Light blue
    'Co': '#66b3ff',  # Blue
    'Ni': '#3399ff',  # Medium blue
    'Cu': '#0080ff',  # Bright blue
    'Zn': '#0066cc',  # Dark blue
    'Ag': '#c0c0c0',  # Silver
    'Au': '#ffd700',  # Gold
    'Pt': '#e6e6e6',  # Platinum
    'Pd': '#cccccc',  # Palladium gray
    'Rh': '#b3b3b3',  # Rhodium gray
    'Ir': '#999999',  # Iridium gray
    'Os': '#808080',  # Osmium gray
    
    # Other metals
    'Al': '#b3b3ff',  # Light purple
    'Pb': '#8080ff',  # Purple
    'Sn': '#4d4dff',  # Dark purple
    'Ti': '#e6ccff',  # Lavender
    'Cr': '#cc99ff',  # Violet
    'Mn': '#b366ff',  # Deep violet
    
    # Non-metals
    'O': '#99ff99',   # Light green
    'C': '#66ff66',   # Green
    'H': '#33cc33',   # Medium green
    'N': '#009900',   # Dark green
    'S': '#ffff99',   # Light yellow
    'P': '#ffff66',   # Yellow
    'Cl': '#ffff33',   # Bright yellow
    'F': '#ffcc00',   # Golden yellow
    
    # Rare earth elements
    'La': '#ff99cc',   # Pink
    'Ce': '#ff66b3',   # Medium pink
    'Pr': '#ff3399',   # Dark pink
    'Nd': '#ff0080',   # Hot pink
    'Sm': '#e60073',   # Deep pink
    'Eu': '#cc0066',   # Very deep pink
    
    # Special case
    'Rest': '#d9d9d9',  # Light gray
}

# Generate colors for elements not in the predefined dictionary
def get_element_color(element_name, idx=None):
    """Return consistent color for an element across all plots."""
    if element_name in ELEMENT_COLORS:
        return ELEMENT_COLORS[element_name]
    
    # Fallback: use a fixed colormap with index
    # This ensures same element always gets same color for same session
    if hasattr(get_element_color, '_color_cache') and element_name in get_element_color._color_cache:
        return get_element_color._color_cache[element_name]
    
    # Initialize cache
    if not hasattr(get_element_color, '_color_cache'):
        get_element_color._color_cache = {}
        get_element_color._next_idx = 0
    
    # Assign a color from a perceptually uniform colormap
    cmap = plt.cm.tab20 # type: ignore
    color = cmap(get_element_color._next_idx % 20)
    get_element_color._color_cache[element_name] = color
    get_element_color._next_idx += 1
    
    return color

def plot_element_entropy_evolution(element_entropy_dfs, elements, plot_total=False, file_path=''):
    """
    Parameters:
    - element_entropy_dfs: dataframe computed by 'compute_element_entropy_across_stages(stage_dfs)'
    - elements: list of element(s) to analyse its RSE evolution
    - plot_total: bool, if True, plots total RSE (sum of all elements) on same y-axis
    - file_path: path to save the figure

    Return:
    - A plot of the evolution of elements' Relative Statistical entropy (RSE) across stages.
    """
    # Convert single element to a list
    if isinstance(elements, str):
        elements = [elements]

    stages = sorted(element_entropy_dfs.keys())

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plot RSE for each element with consistent colors
    for element in elements:
        relative_entropy = []

        for stage in stages:
            try:
                relative_entropy.append(element_entropy_dfs[stage].loc[element, "Relative"])
            except KeyError:
                print(f"Element '{element}' not found in stage {stage}. Available elements: {element_entropy_dfs[stage].index.tolist()}")
                relative_entropy.append(np.nan)

        # Get consistent color
        color = get_element_color(element)
        ax.plot(stages, relative_entropy, marker='s', color=color, linewidth=2, 
                markersize=8, label=f'{element} RSE')

    # Plot total RSE (always black)
    if plot_total:
        total_rse = []
        for stage in stages:
            total_rse.append(element_entropy_dfs[stage]["Relative"].sum())
        
        ax.plot(stages, total_rse, marker='o', color='black', linestyle='--', 
                linewidth=2.5, markersize=8, label='Total RSE (all elements)', zorder=10)

    # Add title and labels
    plt.title(f'Evolution of Relative Statistical Entropy Across Stages', fontsize=14, fontweight='bold')
    ax.set_xlabel('Stage', fontsize=12)
    ax.set_ylabel('Relative Statistical Entropy', fontsize=12)
    ax.legend(loc='upper left', fontsize=10, framealpha=0.9)
    ax.grid(True, alpha=0.3, linestyle='--')

    plt.tight_layout()
    
    plt.savefig(f'{file_path}element_entropy_evolution_{"_".join(elements)}.svg', dpi=200, bbox_inches='tight')
    
    plt.show()

## general

In [ ]:
def compute_all_stages_entropy(stage_dfs):
    """
    Compute Shannon, Rechberger, and Relative entropy for each flow in all stages.
    Uses the compute_stage_entropy function for each stage.

    Parameters:
    - stage_dfs: Dictionary of DataFrames, where each key is a stage name and the value is the stage DataFrame.

    Returns:
    - Dictionary of DataFrames with added entropy columns for each stage.
    """
    stage_dfs_with_entropies = {}

    for stage, df in stage_dfs.items():
        # Compute entropy for this stage
        general_df_with_entropies = compute_stage_entropy(df)
        stage_dfs_with_entropies[stage] = general_df_with_entropies

    return stage_dfs_with_entropies 

def summarize_entropy(all_stages_with_entropies):
    """
    Summarize entropy results across all stages.
    
    Parameters:
    - all_stages_with_entropies: Dictionary of {stage_name: DataFrame} with computed entropy columns
    
    Returns:
    - DataFrame summarizing entropy metrics per stage
    """
    summary = {}
    
    # Fixed: using the parameter name consistently
    for stage, df in all_stages_with_entropies.items():
        total_rechberger = df["Rechberger entropy per flow"].sum()
        total_relative = df["Relative entropy per flow"].sum()
        
        summary[stage] = {
            "Rechberger entropy [bits]": total_rechberger,
            "Relative entropy [normalized amount]": total_relative,
        }
    
    return pd.DataFrame(summary).T  # Transpose for stages as rows

def plot_entropy(summary_df):
    """
    Plot SE and RSE across stages with dual y-axes.
    
    Parameters:
    - summary_df: The entropy summarized results from summarize_entropy()
    
    Returns:
    - The evolution of SE and RSE across the flowsheet's stages
    """
    # summary_df already has stages as rows, metrics as columns
    plot_df = summary_df.copy()
    
    # Sort by stage index (convert to int if needed)
    plot_df = plot_df.sort_index()
    
    # Replace infinities and NaNs
    plot_df = plot_df.replace([np.inf, -np.inf], np.nan).fillna(0)

    fig, ax1 = plt.subplots(figsize=(10, 6))

    # Get x-axis values (stage indices)
    stages = plot_df.index.tolist()
    
    # Plot SE on the left y-axis
    color_se = 'tab:blue'
    ax1.set_xlabel("Stage")
    ax1.set_ylabel("SE [bits]", color=color_se)
    ax1.plot(
        stages,
        plot_df["Rechberger entropy [bits]"],
        marker="o",
        color=color_se,
        label="SE [bits]"
    )
    ax1.tick_params(axis='y', labelcolor=color_se)
    ax1.grid(True)
    
    # Set x-ticks to show all stages
    ax1.set_xticks(stages)

    # Create a second y-axis for RSE
    ax2 = ax1.twinx()
    color_rse = 'tab:red'
    ax2.set_ylabel("RSE [normalised amount]", color=color_rse)
    ax2.plot(
        stages,
        plot_df["Relative entropy [normalized amount]"],
        marker="s",
        color=color_rse,
        label="RSE [normalised amount]"
    )
    ax2.tick_params(axis='y', labelcolor=color_rse)

    # Add title and legend
    plt.title("Entropy Evolution Across Stages")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    plt.savefig(f'{file_path}entropy_evolution_across_stages.svg', dpi=200, bbox_inches='tight')
    plt.show()

# Analyse the SEA

Flow and stage comparison functions reused from `main_sea.ipynb`.

## compare flows between two stages

In [ ]:
import pandas as pd

def compare_stages(df1, df2, key_columns=['flow_name', 'flow_type', 'from_process', 'to_process']):
    """
    Compare two stage DataFrames and classify flows into:
    - Identical flows (same flow_name, flow_type, from_process, to_process, and all element values)
    - Differences (flows unique to df1, unique to df2, or with different values)

    Args:
        df1 (pd.DataFrame): First stage DataFrame.
        df2 (pd.DataFrame): Second stage DataFrame.
        key_columns (list): Columns to use as identifiers for flows.

    Returns:
        dict: A dictionary with keys 'identical', 'only_in_df1', 'only_in_df2', and 'differences'.
    """
    # Merge on key columns to find identical flows (all columns must match)
    merged = pd.merge(df1, df2, on=key_columns, how='outer', suffixes=('_1', '_2'), indicator=True)

    # Identical flows: present in both DataFrames with all values matching
    identical_mask = (merged['_merge'] == 'both')
    for col in df1.columns:
        if col not in key_columns:
            identical_mask &= (merged[f'{col}_1'] == merged[f'{col}_2'])

    identical = merged[identical_mask][key_columns].drop_duplicates().reset_index(drop=True)

    # Flows only in df1
    only_in_df1 = df1[~df1.set_index(key_columns).index.isin(df2.set_index(key_columns).index)].reset_index(drop=True)

    # Flows only in df2
    only_in_df2 = df2[~df2.set_index(key_columns).index.isin(df1.set_index(key_columns).index)].reset_index(drop=True)

    # Flows with differences (same key columns but different values)
    diff_mask = (merged['_merge'] == 'both') & (~identical_mask)
    differences = merged[diff_mask][key_columns].drop_duplicates().reset_index(drop=True)

    return {
        'identical': identical,
        'only_in_df1': only_in_df1,
        'only_in_df2': only_in_df2,
        'differences': differences
    }

## contribution analysis function

In [ ]:
# stage contribution
def compute_rse_differences(stage_entropy_dfs):
    """
    Compute differences in Relative Statistical Entropy (RSE) between consecutive stages.
    
    Parameters:
    - stage_entropy_dfs: Dictionary of {stage_name: DataFrame} with computed entropy columns
    
    Returns:
    - DataFrame with RSE differences between consecutive stages
    """
    diffs = []
    stages = sorted(stage_entropy_dfs.keys())
    
    for i in range(len(stages) - 1):
        n, n1 = stages[i], stages[i+1]
        rse_n = stage_entropy_dfs[n]["Relative entropy per flow"].sum()
        rse_n1 = stage_entropy_dfs[n1]["Relative entropy per flow"].sum()
        delta = float(rse_n1 - rse_n)  # Convert from np.float64 to regular float
        
        diffs.append({
            "From Stage": n,
            "To Stage": n1,
            "Stage Transition": f"{n} → {n1}",
            "RSE_n": rse_n,
            "RSE_n+1": rse_n1,
            "ΔRSE": delta,
            "Absolute ΔRSE": abs(delta)
        })
    
    return pd.DataFrame(diffs)

# flow contribution in stage df and plot
def compare_flows_between_stages(stage_entropy_dfs, stage_a, stage_b):
    """
    Compare flows between two stages. 

    Parameters:
    - Require the stages' entropy variation dfs, and a selection of two stages.
    
    Returns:
    - two DataFrames with flow_name and RSE for stage_a and stage_b.
    """
    df_a = stage_entropy_dfs[stage_a][["flow_name", "Relative entropy per flow"]].copy()
    df_b = stage_entropy_dfs[stage_b][["flow_name", "Relative entropy per flow"]].copy()

    # Group by flow_name in case of duplicates
    df_a = df_a.groupby("flow_name", as_index=False).sum()
    df_b = df_b.groupby("flow_name", as_index=False).sum()

    # Merge them
    merged = pd.merge(df_a, df_b, on="flow_name", how="outer", suffixes=(f"_stage{stage_a}", f"_stage{stage_b}"))
    merged = merged.fillna(0)

    return merged


def plot_flow_rse_differences(merged_df, stage_a, stage_b):
    """
    Plot RSE per flow for two stages side by side.
    """
    flows = merged_df["flow_name"]
    rse_a = merged_df[f"Relative entropy per flow_stage{stage_a}"]
    rse_b = merged_df[f"Relative entropy per flow_stage{stage_b}"]

    # Sort flows by difference
    order = np.argsort(rse_b - rse_a)
    flows_sorted = flows.iloc[order]
    rse_a_sorted = rse_a.iloc[order]
    rse_b_sorted = rse_b.iloc[order]

    x = np.arange(len(flows_sorted))

    plt.figure(figsize=(12, 6))
    plt.bar(x - 0.2, rse_a_sorted, width=0.4, label=f"Stage {stage_a}")
    plt.bar(x + 0.2, rse_b_sorted, width=0.4, label=f"Stage {stage_b}")

    plt.xticks(x, flows_sorted, rotation=90)
    plt.ylabel("Relative Entropy per flow")
    plt.title(f"Flow-level RSE comparison: Stage {stage_a} → {stage_b}")
    plt.legend()
    plt.tight_layout()
    plt.show()


# Section 1 — Robustness to the selection of the original process $p_0$ (SM-3, Figure S2)

The stage construction (process ranking, flow classification, stage creation) is repeated for two different choices of the original process $p_0$, and the resulting SEA curves are compared.

### process ranking

In [ ]:
# step 0: get the possible original processes
original_processes_list = get_original_processes(excel_file_path)

# Step 1: Get all ranking options
all_rankings, process_consumers, process_providers = assign_process_ranks_all_options(
    excel_file_path,
    original_processes_list
)

# Step 2: Display all options and get dataframes
all_rankings, ranking_dfs = print_all_ranking_options(all_rankings)

### process ranking choice

In [ ]:
# ✋ Manual : Print option X ranking DataFrame
X = 0 # X = 0 stands for option 1
print(f"\n=== Option {X} DataFrame ===")
print(ranking_dfs[X]['ranking_df'])
max_rank_per_process = ranking_dfs[X]['max_rank_per_process']
process_ranks = ranking_dfs[X]['process_ranks']
recycling_flows = ranking_dfs[X]["recycling_flows"]

# 🚴 Auto: User selects preferred option
# chosen_original, chosen_ranking = select_ranking_option(all_rankings)
# max_rank_per_process = chosen_ranking["max_rank_per_process"]
# process_ranks = chosen_ranking["process_ranks"]
# recycling_flows = chosen_ranking["recycling_flows"]


In [ ]:
# Figure S2 : the two original process options that are compared
# ✋ Manual : select the two options X (X = 0 stands for option 1)
options_to_compare = [0, 1]

def build_stage_dfs_for_option(X):
    """
    Build the stage_dfs for the ranking option X (X = 0 stands for option 1),
    with the same functions as in main_sea.ipynb.
    """
    max_rank_per_process = ranking_dfs[X]['max_rank_per_process']
    process_ranks = ranking_dfs[X]['process_ranks']

    recycling_flows_df = get_internal_recycling_flows(
        excel_file_path,
        process_consumers,
        max_rank_per_process
    )

    input_env_flows_df = get_input_environment_flows(
        excel_file_path,
        max_rank_per_process
    )

    output_env_flows_df = get_output_environment_flows(
        excel_file_path,
        max_rank_per_process
    )

    intermediate_foreground_flows_df = (
        get_intermediate_foreground_product_flows(
            excel_file_path,
            max_rank_per_process,
            recycling_flows_df
        )
    )

    stage_dfs = build_stage_flow_dataframes(
        excel_file_path,
        process_ranks,
        max_rank_per_process,
        recycling_flows_df,
        input_env_flows_df,
        output_env_flows_df,
        intermediate_foreground_flows_df
    )

    return stage_dfs

In [ ]:
def plot_entropy_comparison(summaries, labels, title):
    """
    Plot the evolution of SE and RSE across stages for several SEA stage constructions.

    Parameters:
    - summaries: list of entropy summary DataFrames (from summarize_entropy())
    - labels: list of labels, one per summary
    - title: figure title
    """
    fig, (ax_se, ax_rse) = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10.colors
    all_stages = sorted({s for sdf in summaries for s in sdf.index})

    for k, (summary_df, label) in enumerate(zip(summaries, labels)):
        plot_df = summary_df.copy().sort_index()
        plot_df = plot_df.replace([np.inf, -np.inf], np.nan).fillna(0)
        color = colors[k % len(colors)]
        ax_se.plot(plot_df.index, plot_df["Rechberger entropy [bits]"],
                   marker="o", color=color, linewidth=2, label=label)
        ax_rse.plot(plot_df.index, plot_df["Relative entropy [normalized amount]"],
                    marker="s", color=color, linewidth=2, label=label)

    for ax, ylabel in [(ax_se, "SE [bits]"), (ax_rse, "RSE [normalised amount]")]:
        ax.set_xlabel("Stage")
        ax.set_ylabel(ylabel)
        ax.set_xticks(all_stages)
        ax.grid(True)
        ax.legend()

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(f'{file_path}entropy_evolution_across_stages_options_comparison.svg', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
# Build the SEA for each original process option and summarise the entropy results
option_stage_dfs = {}
option_summaries = {}

for X in options_to_compare:
    print(f"\n=== Option {X} : original process p0 = '{ranking_dfs[X]['original_process']}' ===")
    stage_dfs = build_stage_dfs_for_option(X)
    option_stage_dfs[X] = stage_dfs

    all_stage_s_dfs = compute_all_stages_entropy(stage_dfs)
    summary_df = summarize_entropy(all_stage_s_dfs)
    option_summaries[X] = summary_df
    print(summary_df)

In [ ]:
# Figure S2 : overlay of the entropy evolutions for the two original process options
labels = [f"p0 = {ranking_dfs[X]['original_process']}" for X in options_to_compare]

plot_entropy_comparison(
    [option_summaries[X] for X in options_to_compare],
    labels,
    "Entropy Evolution Across Stages for Different Original Processes"
)

# Side-by-side comparison of the RSE per stage
comparison_df = pd.DataFrame({
    f"Option {X}: {ranking_dfs[X]['original_process']}": option_summaries[X]["Relative entropy [normalized amount]"]
    for X in options_to_compare
})
print("\n=== RSE per stage, per original process option ===")
print(comparison_df)

# Final RSE reached by each stage construction
final_rse = pd.DataFrame({
    "Original process p0": [ranking_dfs[X]['original_process'] for X in options_to_compare],
    "Final RSE": [option_summaries[X]["Relative entropy [normalized amount]"].iloc[-1] for X in options_to_compare],
    "Final SE [bits]": [option_summaries[X]["Rechberger entropy [bits]"].iloc[-1] for X in options_to_compare],
})
final_rse["Relative difference on final RSE"] = final_rse["Final RSE"] / final_rse["Final RSE"].iloc[0] - 1
print("\n=== Final entropy values per original process option ===")
print(final_rse)

# Section 2 — Robustness to the selection of the target elements (SM-3, Figure S3)

For the stage construction of option 1 (`X = 0`), the SEA is recomputed for different lists of target elements, the other elements being diluted in a *Rest* fraction (see `group_elements_of_interest`).

In [ ]:
# Reference stage construction: original process of option 1 (X = 0)
X = 0
stage_dfs = option_stage_dfs[X]

all_elements = get_unique_elements(stage_dfs)
print(f"Elements present in the flowsheet: {all_elements}")

# Figure S3 : the target element selections that are compared
element_selections = {
    "all elements (detailed)": all_elements,
    "Li, Ni, Mn, Co": ["Li", "Ni", "Mn", "Co"],
    "Li, Ni": ["Li", "Ni"],
    "Li": ["Li"],
}

In [ ]:
# Build the SEA for each target element selection and summarise the entropy results
selection_summaries = {}

for label, elements_of_interest in element_selections.items():
    print(f"\n=== Target elements: {label} ===")
    if elements_of_interest == all_elements:
        # No grouping: the full detailed element columns are kept
        restricted_stage_dfs = stage_dfs
    else:
        restricted_stage_dfs = group_elements_of_interest(stage_dfs, elements_of_interest)

    all_stage_s_dfs = compute_all_stages_entropy(restricted_stage_dfs)
    summary_df = summarize_entropy(all_stage_s_dfs)
    selection_summaries[label] = summary_df
    print(summary_df)

In [ ]:
# Figure S3 : overlay of the RSE evolutions for the different target element selections
plot_entropy_comparison(
    list(selection_summaries.values()),
    list(selection_summaries.keys()),
    "Entropy Evolution Across Stages for Different Target Element Selections"
)

# Final RSE reached for each target element selection
selection_final_df = pd.DataFrame({
    "Target elements": list(selection_summaries.keys()),
    "Final RSE": [s["Relative entropy [normalized amount]"].iloc[-1] for s in selection_summaries.values()],
    "Final SE [bits]": [s["Rechberger entropy [bits]"].iloc[-1] for s in selection_summaries.values()],
})
print("\n=== Final entropy values per target element selection ===")
print(selection_final_df)

# Section 3 — Product contributions of the first and last stage (`X = 0`)

For the stage construction of option 1 (`X = 0`), the product flow contributions to the RSE are compared between the **first** and the **last** stage. All the natural environment output flows are grouped under an artificial flow named `elementary_flows`, and the cut-off waste input LiBs (`f_0-1_eol-libs`) is kept explicit.

In [ ]:
def group_elementary_output_flows(stage_dfs, grouped_flow_name="elementary_flows"):
    """
    Group, within each stage, all the natural environment output flows (flow_type == 'elementary')
    under a single artificial flow. The grouped flow gets the summed mass and the mass-weighted
    element concentrations of the grouped flows. All the other flows (products, including the
    cut-off waste input LiBs) are kept unchanged.

    Parameters:
    - stage_dfs: Dictionary of stage DataFrames.
    - grouped_flow_name: Name of the artificial flow that gathers the elementary output flows.

    Returns:
    - Dictionary of stage DataFrames with the elementary output flows grouped.
    """
    modified_stage_dfs = {}

    for stage, df in stage_dfs.items():
        if df.empty:
            modified_stage_dfs[stage] = df
            continue

        df = df.copy()
        element_cols = [col for col in df.columns if is_chemical_element(col)]

        mask = df["flow_type"].astype(str).str.strip().str.lower() == "elementary"
        grouped = df[mask]
        kept = df[~mask].copy()

        if not grouped.empty:
            total_mass = grouped["mass-FU_kg-eq"].sum()
            row = {
                "stage": stage,
                "flow_name": grouped_flow_name,
                "flow_type": "elementary",
                "from_process": "foreground_processes",
                "to_process": "natural environment",
                "mass-FU_kg-eq": total_mass,
            }
            for element in element_cols:
                weighted = (grouped[element].fillna(0) * grouped["mass-FU_kg-eq"]).sum()
                row[element] = weighted / total_mass if total_mass != 0 else 0
            kept = pd.concat([kept, pd.DataFrame([row])], ignore_index=True)

        modified_stage_dfs[stage] = kept.sort_values("flow_name").reset_index(drop=True)

    return modified_stage_dfs

In [ ]:
# Group the natural environment output flows and display the first and last stages
grouped_stage_dfs = group_elementary_output_flows(stage_dfs)

first_stage = min(grouped_stage_dfs.keys())
last_stage = max(grouped_stage_dfs.keys())

print(f"=== First stage (stage {first_stage}) flows ===")
print(grouped_stage_dfs[first_stage][["flow_name", "flow_type", "from_process", "to_process", "mass-FU_kg-eq"]])

print(f"\n=== Last stage (stage {last_stage}) flows ===")
print(grouped_stage_dfs[last_stage][["flow_name", "flow_type", "from_process", "to_process", "mass-FU_kg-eq"]])

In [ ]:
# Compare the product flow contributions to the RSE between the first and the last stage
all_stage_s_dfs = compute_all_stages_entropy(grouped_stage_dfs)

merged_df = compare_flows_between_stages(all_stage_s_dfs, first_stage, last_stage)
merged_df = merged_df.sort_values(f"Relative entropy per flow_stage{first_stage}", ascending=False)

print(f"=== Flow contributions to the RSE: Stage {first_stage} vs Stage {last_stage} ===")
print(merged_df)

# Figure : flow-level RSE comparison between the first and the last stage
plot_flow_rse_differences(merged_df, first_stage, last_stage)

# 🔚 end